In [1]:
%matplotlib widget

from blackjack_cpp import ProbabilisticRankShoe, RandomSampler


from blackjack.blackjack_round import BJRound, BJStage, BJRules
from blackjack.actions import PlayerAction, DealerAction
from blackjack.cards import Card, Rank
import numpy as np
import time
from datetime import datetime
import os
import tqdm
from collections import deque
# from blackjack.shoe import ProbabilisticRankShoe
import time
from blackjack.floor_ceil_node import (
    FloorCeilNode, SplitNode, DecisionNode, DealerCheckBJNode
)
from blackjack.dealer_sim import run_dealer_cards_simulation
import datetime
import matplotlib.pyplot as plt
from blackjack.tree_utils import iterate_nodes_by_levels
import os
from blackjack import dealer_sim

from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing
from dataclasses import asdict

    
# Import inside worker to ensure proper initialization
from blackjack.abstract_node import ValueNode
from blackjack_cpp import ProbabilisticRankShoe
from blackjack.blackjack_round import BJRound, BJRules
from blackjack.floor_ceil_node import DecisionNode

RandomSampler.reset_global_seed_generator(0)


In [2]:
ranks_cpp = [7, 2, 5, 8, 2, 5, 4, 10, 9, 7, 9, 4, 4, 10, 2, 11, 10, 7, 10, 10, 8, 8, 9, 4, 5, 5, 8, 10, 10, 2, 2, 10, 11, 8, 9, 10, 10, 5, 10, 3, 4, 3, 5, 10, 4, 9, 9, 10, 9, 10, 10, 4, 2, 5, 9, 10, 9, 10, 11, 10, 7, 3, 6, 9, 6, 3, 3, 10, 7, 4, 10, 2, 6, 11, 6, 5, 10, 8, 6, 7, 3, 10, 10, 7, 10, 5, 7, 10, 2, 11, 7, 5, 4, 6, 10, 11, 10, 6, 2, 6, 11, 3, 8, 10, 8, 5, 10, 3, 10, 9, 5, 10, 9, 9, 10, 2, 10, 10, 7, 10, 3, 11, 9, 10, 3, 11, 10, 8, 11, 2, 10, 9, 11, 11, 3, 4, 10, 3, 11, 10, 10, 7, 10, 10, 9, 5, 2, 9, 10, 5, 10, 10, 4, 7, 4, 4, 2, 10, 10, 8, 10, 10, 6, 7, 8, 8, 11, 6, 10, 8, 5, 10, 8, 10, 10, 10, 11, 10, 10, 10, 3, 6, 11, 2, 6, 8, 3, 10, 8, 5, 10, 10, 10, 10, 10, 7, 11, 7, 4, 9, 10, 2, 10, 10, 7, 7, 5, 6, 3, 7, 10, 6, 11, 3, 10, 3, 4, 9, 2, 11, 9, 7, 11, 3, 6, 6, 9, 2, 9, 10, 4, 8, 10, 7, 10, 10, 10, 9, 3, 2, 4, 8, 5, 2, 7, 8, 4, 11, 3, 8, 6, 10, 10, 10, 6, 10, 3, 3, 2, 8, 10, 2, 6, 9, 10, 8, 8, 4, 10, 2, 9, 6, 10, 10, 5, 6, 2, 11, 10, 8, 6, 5, 2, 10, 10, 11, 7, 11, 5, 10, 7, 10, 10, 10, 3, 10, 11, 7, 3, 3, 4, 3, 7, 11, 9, 11, 2, 4, 4, 5, 3, 10, 10, 10, 9, 10, 10, 10, 10, 8, 9, 10, 11, 3, 10, 10, 4, 2, 3, 5, 8, 6, 10, 7, 2, 10, 5, 4, 6, 7, 5, 2, 9, 10, 10, 4, 7, 10, 10, 5, 10, 2, 11, 9, 10, 10, 4, 5, 8, 7, 8, 10, 9, 9, 10, 7, 10, 4, 11, 11, 10, 10, 3, 4, 6, 10, 4, 5, 10, 2, 5, 5, 10, 6, 6, 8, 4, 10, 3, 6, 8, 10, 8, 10, 6, 5, 10, 6, 8, 7, 10, 7, 2, 6, 10, 11, 4, 10, 3, 4, 10, 6, 10, 5, 10, 2]
ranks_cpp[:10]

[7, 2, 5, 8, 2, 5, 4, 10, 9, 7]

In [3]:
RandomSampler.reset_global_seed_generator(0)

In [4]:
shoe = ProbabilisticRankShoe(8)
ranks_py = [shoe.sample_and_burn_rank() for _ in range(8*52)]
ranks_py[:10]

[7, 2, 5, 8, 2, 5, 4, 10, 9, 7]

In [5]:
for i in range(100):
    if ranks_cpp[i] != ranks_py[i]:
        print(i, ranks_cpp[i], ranks_py[i])

In [6]:
import pickle

# Test pickling and unpickling a shoe
RandomSampler.reset_global_seed_generator(42)

# Create a shoe and modify its state
shoe = ProbabilisticRankShoe(6)
shoe.burn_rank_value(10)
shoe.burn_rank_value(10)
shoe.burn_rank_value(5)
shoe.lock_dealer_card_not_ace()

# Sample some values to advance RNG state
samples_before = [shoe.sample_rank() for _ in range(5)]

print("Before pickling:")
print(f"  Dealer locked value: {shoe.dealer_card_locked_value()}")
print(f"  Number of 10s: {shoe.get_number_of_rank_cards(10)}")
print(f"  Number of 5s: {shoe.get_number_of_rank_cards(5)}")
print(f"  Samples: {samples_before}")

# Pickle and unpickle
pickled = pickle.dumps(shoe)
restored_shoe = pickle.loads(pickled)

# Sample from restored shoe - should produce same sequence as original would
samples_after_original = [shoe.sample_and_burn_rank() for _ in range(5)]
samples_after_restored = [restored_shoe.sample_and_burn_rank() for _ in range(5)]

print("\nAfter unpickling:")
print(f"  Dealer locked value: {restored_shoe.dealer_card_locked_value()}")
print(f"  Number of 10s: {restored_shoe.get_number_of_rank_cards(10)}")
print(f"  Number of 5s: {restored_shoe.get_number_of_rank_cards(5)}")
print(f"  Samples from original: {samples_after_original}")
print(f"  Samples from restored: {samples_after_restored}")
print(f"  RNG states match: {samples_after_original == samples_after_restored}")

Before pickling:
  Dealer locked value: 11
  Number of 10s: 94
  Number of 5s: 23
  Samples: [6, 5, 6, 5, 10]

After unpickling:
  Dealer locked value: 11
  Number of 10s: 90
  Number of 5s: 23
  Samples from original: [10, 10, 10, 9, 10]
  Samples from restored: [10, 10, 10, 9, 10]
  RNG states match: True


In [ ]:



def build_root_node(bj_round, shoe, sim_depth, sim_algo):
    stage = bj_round.get_stage()

    if stage == BJStage.DEALER_CHECK_BJ:
        # dealer checks blackjack with ten
        # insurance not offered
        root_node = DealerCheckBJNode(
            bj_round,
            shoe,
            max_hand_size_full_enum=1,
            took_insurance=False,
            insurance_offered=False,
            dealer_sim_depth=sim_depth,
            sim_algo=sim_algo
        )
    elif stage == BJStage.DEALER_CARD \
        and len(bj_round.player_hands) == 1 \
        and bj_round.player_hands[0].is_natural_blackjack():
        # player has blackjack, insurance not offered - go to dealer card immediately
        root_node = ValueNode(
            bj_round.bet_unit * bj_round.rules.natural_blackjack_payout,
            bj_round=bj_round,
            shoe=shoe
        )
    else:
        root_node = DecisionNode(
            bj_round,
            shoe,
            max_hand_size_full_enum=1,
            dealer_sim_depth=sim_depth,
            sim_algo=sim_algo
        )
            
    return root_node



def is_waiting_for_decision(root_node):
    if isinstance(root_node, DecisionNode):
        if root_node.bj_round.get_stage() == BJStage.PLAYER_ACTION:
            return root_node.decision_choice is None
        else:  # insurance decision
            if root_node.decision_choice is not None:
                insurance_decision_child = root_node.get_decision_choice_child()
                return is_waiting_for_decision(insurance_decision_child)
            else:
                return True
            
    elif isinstance(root_node, DealerCheckBJNode):
        return is_waiting_for_decision(
            root_node.children[root_node.dealer_no_bj_child_idx]
        )
    else:
        return False
    
def build_tree(root_node, gap_target):
    if isinstance(root_node, ValueNode):
        return
    
    gap_target_unit = gap_target * root_node.bj_round.bet_unit
    root_node.build_tree()
    for i in range(100):
        root_node.convert_to_full_up_to_depth(depth=i)
        
        value_gap_ok = (root_node.get_ceil_value() - root_node.get_floor_value()) < gap_target_unit
        if value_gap_ok and not is_waiting_for_decision(root_node):
            return





In [8]:
rules = BJRules(
    dealer_checks_blackjack=True,
    dealer_hits_soft_17=False,
    allow_late_surrender=False,
    allow_early_surrender_on_ten=False,
    allow_early_surrender_on_ace=False,
    allow_early_surrender_on_all=False,
    dealer_shows_card_on_surrender=False,
    allow_insurance_vs_ace=True,
    natural_blackjack_payout=3/2,
    surrender_payout=1/2,
    insurance_payout=2/1,
    max_splits_allowed=1,
    allow_action_on_split_aces=True,
    allow_double_after_split=True,
    allow_double_on_soft=True,
    allow_split_different_tens=True
)

In [9]:
def generate_initial_rounds():
    for player_card_0 in range(2, 12):
        for player_card_1 in range(player_card_0, 12):
            for dealer_upcard in range(2, 12):
                n_count = 1 if player_card_0 == player_card_1 else 2
                yield player_card_0, player_card_1, dealer_upcard, n_count

In [10]:
def measure_runtime(bj_round, shoe):

    runtime_data = {}
    RandomSampler.reset_global_seed_generator()

    for depth in [2, 3, 4, 5, 6, 7]:
        for algo in ["combo", "recursive"]:
            root_node_timing = build_root_node(bj_round, shoe, depth, algo)
            t0 = time.time()
            
            if not isinstance(root_node_timing, ValueNode):
                root_node_timing.build_tree()
                for i in range(100):
                    t0 = time.time()
                    root_node_timing.convert_to_full_up_to_depth(depth=i)
                    t1 = time.time()
                    if root_node_timing.get_ceil_value() - root_node_timing.get_floor_value() < 0.01:
                        break

            t1 = time.time()
            gap = (root_node_timing.get_floor_value(), root_node_timing.get_ceil_value())
            value = root_node_timing.get_value()
            runtime = t1 - t0

            runtime_data[(depth, algo)] = (runtime, gap, value)
    
    return runtime_data


In [11]:
def get_start_shoe():
    shoe = ProbabilisticRankShoe(6)
    # for low_count in range(2, 7):
    #     shoe.set_number_of_rank_cards(low_count, 1)
    return shoe


def get_ev_single_round_args(args):
    """Worker function that reconstructs node and computes value."""
    p0, p1, d, gap_target, rules_dict, sim_depth, algo = args
    
    # Reconstruct rules and objects
    rules = BJRules(**rules_dict)
    shoe = get_start_shoe()
    bj_round = BJRound(rules)
    bj_round.start_round(100)
    
    for c in [p0, p1, d]:
        bj_round.take_card(c)
        shoe.burn_rank_value(d)
    
    root_node = build_root_node(bj_round, shoe, sim_depth, algo)
    return get_ev_single_round_node(root_node, gap_target)


def measure_runtime_args(args):
    p0, p1, d, rules_dict = args    
    # Reconstruct rules and objects
    rules = BJRules(**rules_dict)
    shoe = get_start_shoe()
    bj_round = BJRound(rules)
    bj_round.start_round(100)
    for c in [p0, p1, d]:
        bj_round.take_card(c)
        shoe.burn_rank_value(d)
    return measure_runtime(bj_round, shoe)


def get_ev_single_round_node(root_node, gap_target):
    build_tree(root_node, gap_target)
    return root_node, root_node.get_value(), root_node.get_floor_value(), root_node.get_ceil_value()


In [12]:
nodes = []
probs = []
runtime_tasks = []
ev_tasks = []

sim_depth = 5
gap_target = 0.01
rules_dict = asdict(rules)
algo = "recursive"

for p0, p1, d, n in generate_initial_rounds():
    bj_round = BJRound(rules)
    bj_round.start_round(100)
    shoe = get_start_shoe()
    prob = 1
    for c in [p0, p1, d]:
        prob_c_dict = shoe.get_rank_value_probabilities()
        prob *= prob_c_dict.get(c, 0)
        if prob == 0:
            break
        bj_round.take_card(c)
        shoe.burn_rank_value(c)
    
    if prob == 0:
        continue
    
    root_node = build_root_node(
        bj_round,
        shoe,
        sim_depth,
        algo
    )
    nodes.append(root_node)
    probs.append(prob * n)
    ev_tasks.append((p0, p1, d, gap_target, rules_dict, sim_depth, algo))
    runtime_tasks.append((p0, p1, d, rules_dict,))

print(np.sum(probs))

1.0


In [13]:
# results_by_nodes = []
# for node in tqdm.tqdm(nodes):
#     result = process_single_round_node(node, gap_target)
#     results_by_nodes.append(result)

In [14]:

max_workers = multiprocessing.cpu_count()

results_by_nodes = [None] * len(runtime_tasks)

with ProcessPoolExecutor(max_workers=max_workers) as executor:
    future_to_idx = {
        executor.submit(measure_runtime_args, task): idx 
        for idx, task in enumerate(runtime_tasks)
    }
    
    # future_to_idx = {
    #     executor.submit(process_single_round_node, node, gap_target): idx
    #     for idx, node in enumerate(nodes)
    # }

    for future in tqdm.tqdm(as_completed(future_to_idx), total=len(runtime_tasks)):
        idx = future_to_idx[future]
        results_by_nodes[idx] = future.result()

100%|██████████| 550/550 [2:22:32<00:00, 15.55s/it]  


In [15]:
results_by_nodes_clean = [r for r in results_by_nodes if r is not None]

In [16]:
results_by_nodes_clean

[{(2, 'combo'): (14.185515880584717,
   (-7.525971796658232, -7.524549512775623),
   -7.525971796658234),
  (2, 'recursive'): (13.349622011184692,
   (np.float64(-9.09443509945246), np.float64(-9.093988539838538)),
   np.float64(-9.094419794403047)),
  (3, 'combo'): (29.261139631271362,
   (-8.04590405972901, -8.045404152443929),
   -8.045900590189854),
  (3, 'recursive'): (52.38828444480896,
   (np.float64(-8.541807339841457), np.float64(-8.5413150211272)),
   np.float64(-8.541803130744947)),
  (4, 'combo'): (63.08052611351013,
   (-8.302567930745127, -8.30211166954899),
   -8.302563788598645),
  (4, 'recursive'): (166.68086123466492,
   (np.float64(-8.28070220684767), np.float64(-8.280227663842467)),
   np.float64(-8.280681090464276)),
  (5, 'combo'): (98.10129642486572,
   (-8.26822469512053, -8.267775905896881),
   -8.268217218106846),
  (5, 'recursive'): (342.21098470687866,
   (np.float64(-8.268120448619786), np.float64(-8.26765427210573)),
   np.float64(-8.268102724882095)),
  (

In [20]:
import pickle

with open("logs/runtime_results.pkl", "wb") as f:
    pickle.dump(results_by_nodes, f)

In [18]:
r = results_by_nodes_clean[0]

In [19]:
_, (floor_7r, ceil_7r) = r[(7, "recursive")]
_, (floor_7c, ceil_7c) = r[(7, "combo")]
target = (floor_7r + ceil_7r + ceil_7r + floor_7c) / 4

ValueError: too many values to unpack (expected 2)

In [ ]:
target

In [ ]:
for (depth, algo), (runtime, (floor, ceil)) in r.items():
    mid = (floor + ceil) / 2
    print(depth, algo, mid - target)

In [ ]:
r

In [ ]:
ev_mean = np.sum(np.array(ev_by_nodes) * np.array(probs)[:, np.newaxis], axis=0)

In [ ]:
ev, ev_min, ev_max = ev_mean

In [ ]:
ev_mean

In [ ]:
print(f"ev = {ev:.3f}, ev_min = {ev_min:.3f}, ev_max = {ev_max:.3f}")

In [ ]:
for i, (node, ev, ev_min, ev_max) in enumerate(results_by_nodes):
    print("=" * 20)
    print("Idx = ", i)
    node.bj_round.last_action = None
    node.bj_round.last_card = None
    print(str(node.bj_round))
    print(f"value, value_min, value_max = {ev:.2f}, {ev_min:.2f}, {ev_max:.2f}")
    if isinstance(node, DealerCheckBJNode):
        node = node.children[node.dealer_no_bj_child_idx]
        
    if isinstance(node, DecisionNode):
        print(f"best action: {node.decision_choice}")
        if node.decision_choice is None:
            print(f"possibilities: {[a.value for a in node.possible_actions]}")
        elif node.bj_round.get_stage() == BJStage.PLAYER_OFFERED_INSURANCE and not node.bj_round.player_hands[0].is_natural_blackjack():
            child: DealerCheckBJNode = node.get_decision_choice_child()
            decision_grandchild: DecisionNode = child.children[child.dealer_no_bj_child_idx]
            action = decision_grandchild.decision_choice
            print(f"next action: {action}")
    print("=" * 20)

In [ ]:
result_of_interest = results_by_nodes[278]
node_oi, ev_oi, ev_min_oi, ev_max_oi = result_of_interest
print(str(node_oi.bj_round))

In [ ]:
print(str(node_oi.bj_round))
print()
print_tree(node_oi, n_levels=3)